In [6]:
!pip -q install datasets

import re
import cupy as xp
from collections import Counter
from datasets import load_dataset
from cupyx import scatter_add
# --- download TinyStories and take a slice ---
N_STORIES = 80_000
ds = load_dataset("roneneldan/TinyStories", split="train")
stories = ds[:N_STORIES]["text"]
print(f"{len(stories):,} stories")

# --- tokenize: lowercase, words (apostrophes kept) or single punctuation ---
token_re = re.compile(r"[a-z']+|[^\sa-z']")
def tokenize(s):
    return token_re.findall(s.lower())

EOS = "<EOS>"   # marks story boundaries
UNK = "<UNK>"   # stand-in for out-of-vocab words
PAD = "<PAD>"   # blank context before each story
BLOCK_SIZE = 16

all_tokens = []
for s in stories:
    all_tokens.extend([PAD] * BLOCK_SIZE)   # 16 pads: every story starts from blank
    all_tokens.extend(tokenize(s))
    all_tokens.append(EOS)
print(f"{len(all_tokens):,} tokens in the stream")

# --- build the vocab: top VOCAB_SIZE most frequent tokens ---
VOCAB_SIZE = 3000
counts = Counter(all_tokens)
most_common = [tok for tok, _ in counts.most_common(VOCAB_SIZE)]
vocab = [UNK, EOS, PAD] + [t for t in most_common if t not in (UNK, EOS, PAD)]
vocab = vocab[:VOCAB_SIZE]
stoi = {t: i for i, t in enumerate(vocab)}   # UNK=0, EOS=1, PAD=2
itos = {i: t for t, i in stoi.items()}
vocab_size = len(vocab)

# --- coverage check ---
covered = sum(c for t, c in counts.items() if t in stoi)
print(f"vocab size: {vocab_size}, coverage: {100 * covered / len(all_tokens):.2f}%")

# --- tokens -> ids, straight to GPU ---
unk_id = stoi[UNK]
data = xp.array([stoi.get(t, unk_id) for t in all_tokens], dtype=xp.int64)

# --- sliding windows via the index-grid trick ---
n_windows = len(data) - BLOCK_SIZE
offsets = xp.arange(n_windows)[:, None] + xp.arange(BLOCK_SIZE)[None, :]
X = data[offsets]           # (n_windows, 16)
Y = data[BLOCK_SIZE:]       # (n_windows,)

# --- drop windows whose TARGET is <PAD>: never teach the model to predict padding ---
pad_id = stoi[PAD]
keep = Y != pad_id
X, Y = X[keep], Y[keep]

# --- train / val split ---
n = int(0.9 * len(X))
Xtr, Ytr = X[:n], Y[:n]
Xva, Yva = X[n:], Y[n:]
print(f"train windows: {len(Xtr):,}   val windows: {len(Xva):,}")

# --- sanity check: decode one window back to words ---
i = 54321
print([itos[int(j)] for j in X[i]], "->", itos[int(Y[i])])

ModuleNotFoundError: No module named 'cupy'

In [ ]:
emb = (xp.random.randn(vocab_size, 32) * 0.1).astype(xp.float32)

In [ ]:
class Affine:
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def forward(self, x):
        return x @ self.weight.T + self.bias



In [ ]:
class Sigmoid:
    def __init__(self):
        pass

    def forward(self, x):
        return 1 / (xp.exp(-x) + 1)


In [ ]:
class ReLU:
  def __init__(self):
    pass

  def forward(self, x):
    return xp.maximum(0, x)

  def __call__(self, x):
    return self.forward(x)

In [ ]:
class Softmax:
  def __init__(self):
    pass

  def forward(self, x):
    e = xp.exp(x - xp.max(x, axis=1, keepdims=True))
    return e / xp.sum(e, axis=1, keepdims=True)



In [ ]:
import math
def get_lr(i, steps, lr_max=0.5, lr_min=0.02):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * i / steps))

In [ ]:
def top_k(q, k=40):
    idx = xp.argsort(q)[-k:]
    p = xp.zeros_like(q)
    p[idx] = q[idx]
    return p / p.sum()

In [ ]:
class MLP_slm:
    def __init__(self, arc, out):
        self.g = [None] * len(arc)
        for i in range(len(arc)):
            if(i != len(arc) - 1):
                self.g[i] = [(xp.random.randn(arc[i + 1], arc[i]) * xp.sqrt(1/arc[i])).astype(xp.float32), xp.zeros((1, arc[i + 1]), dtype=xp.float32)]
            else:
                self.g[i] = [(xp.random.randn(out, arc[i]) * xp.sqrt(1/arc[i])).astype(xp.float32), xp.zeros((1, out), dtype=xp.float32)]
        self.update()
    def forward(self, x):
        for layer in self.layers:
            for module in layer:
                x =  module.forward(x)
        return x
    def update(self):
        self.n = len(self.g)
        self.affine = [None] * self.n
        self.activation = [None] * self.n
        self.layer = [None] * self.n
        self.layers = [None] * self.n

        for i in range(self.n):
            self.affine[i] = Affine(self.g[i][0], self.g[i][1])
            if(i != self.n - 1):
                self.activation[i] = ReLU()
            else:
                self.activation[i] = Softmax()
            self.layer[i] = [self.affine[i], self.activation[i]]
            self.layers[i] = (self.layer[i])



    def gd(self, unembX, y_train, lr, emb,B=128):
        idx = xp.random.randint(0, len(unembX), B)
        X_train = emb[unembX[idx]].reshape(B, -1)
        y_ids = y_train[idx]
        return self.dgd(X_train, y_ids, lr, emb, unembX[idx])


    def train(self, X_train, y_train, epochs, emb):
        for i in range(epochs):
            self.gd(X_train, y_train, get_lr(i, epochs), emb)


    def test(self, X_test, y_test):
        preds = xp.argmax(self.forward(X_test), axis=1)
        return float((preds == y_test).mean())



    def dgd(self, x, y_ids, lr, emb, Xb):
        xc = x
        n = len(x)
        ins = []
        outs = [0]
        for layer in self.layers:
            ins.append(x)
            for module in layer:
                x =  module.forward(x)
            outs.append(x)
        rolling = x.copy()
        rolling[xp.arange(n), y_ids] -= 1
        for i in range(self.n - 1, -1, -1):
            self.g[i][1] -= lr / n * rolling.sum(axis=0)
            ww = self.g[i][0].copy()
            self.g[i][0] -= lr / n * (rolling.T @  ins[i])
            rolling = rolling @ ww
            if i > 0:
                rolling *= (outs[i] > 0)

        dC = xp.zeros_like(emb)
        scatter_add(dC, Xb, rolling.reshape(n, BLOCK_SIZE, 32))
        emb -= lr / n * dC

        return rolling



    def __call__(self, x):
        return self.forward(x)

In [ ]:
model = MLP_slm([512, 256], vocab_size)
model.train(Xtr, Ytr, 1000000, emb)

In [ ]:
for i in range(1):
    st = ''
    s = [2] * 16
    k = 80
    while True:
        x = model.forward(emb[xp.asarray(s)].reshape(1, -1))
        #print(x)
        q = xp.asnumpy(x)[0]
        q = q / q.sum()

        p = top_k(q, k)
                                           # your (vocab_size,) probabilities
        l = int(xp.searchsorted(xp.cumsum(p), xp.random.rand() * p.sum()))
        if k > 20:
            k -= 3
        if l == 1:
            break
        if(l < 3):
            break
        s = s[1:] + [l]
        st += itos[l]
        st += ' '
    if(len(st) >= 3):
        print(st)


In [ ]:
N_EVAL, CHUNK = 20000, 4096
correct, total_loss, count = 0, 0.0, 0

for i in range(0, N_EVAL, CHUNK):
    Xb = Xva[i:i+CHUNK]
    Yb = Yva[i:i+CHUNK]
    probs = model.forward(emb[Xb].reshape(len(Xb), -1))
    correct += int((xp.argmax(probs, axis=1) == Yb).sum())
    total_loss += float(-xp.log(probs[xp.arange(len(Yb)), Yb] + 1e-9).sum())
    count += len(Yb)

print(f"accuracy: {100 * correct / count:.2f}%")
print(f"val loss: {total_loss / count:.3f}   (random = {float(xp.log(vocab_size)):.2f})")

In [ ]:
def eval_split(Xs, Ys, n_eval=20000, chunk=4096):
    correct, total_loss, count = 0, 0.0, 0
    for i in range(0, n_eval, chunk):
        Xb, Yb = Xs[i:i+chunk], Ys[i:i+chunk]
        probs = model.forward(emb[Xb].reshape(len(Xb), -1))
        correct += int((xp.argmax(probs, axis=1) == Yb).sum())
        total_loss += float(-xp.log(probs[xp.arange(len(Yb)), Yb] + 1e-9).sum())
        count += len(Yb)
    return total_loss / count, 100 * correct / count

tr_loss, tr_acc = eval_split(Xtr, Ytr)
va_loss, va_acc = eval_split(Xva, Yva)
print(f"train: loss {tr_loss:.3f}, acc {tr_acc:.2f}%")
print(f"val:   loss {va_loss:.3f}, acc {va_acc:.2f}%")

In [ ]:
print(model.g[0][0].dtype, model.g[0][1].dtype, emb.dtype)

In [ ]:
from cupyx.profiler import benchmark
print(benchmark(lambda: model.gd(Xtr, Ytr, 0.5, emb, 1280), n_repeat=50))